# 2 · Model, training and evaluation

A **U-Net** (CNN encoder + decoder + 1×1 output convolution) for 6-class segmentation of single raw Sentinel-1 scenes.
All variants were trained with `train/train.py` on GPU 0 of the university server (`train/run_experiments.sh`); this
notebook shows their training logs and curves, evaluates them on the **130 test places** and compares them.
Set `RUN_TRAINING = True` below to train E1 again inside this notebook (about 25 minutes on a GPU).

### How to read this notebook

1. **The network** (2.1): its layers and one prediction traced from input to map.
2. **The variants** (2.2): 11 trainings, each changing one thing, to answer one question each.
3. **Training logs and curves** (2.3–2.4): how training went (notebook 04 shows this in much more detail).
4. **Test results** (2.5): scores on 130 places the network never saw, per class, region and settlement type.
5. **From scenes to monthly maps** (2.6), **learning curve** (2.7), **final setting** (2.8–2.9).
6. **Example predictions** (2.10).

An important rule throughout: **compare variants only on the same labels and the same test places**. Early in the
project two variants were compared against different labels, and a difference that was really in the labels looked
like a model improvement (Project Log).

In [ ]:
import json, sys, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display
warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve().parent                  # the s1-lulc folder
sys.path[:0] = [str(ROOT / "data"), str(ROOT / "train"), str(ROOT / "prototype")]
DATA, PREP, RUNS = ROOT / "data" / "tiles_v0", ROOT / "data" / "tiles_v0" / "prepared", ROOT / "train" / "runs"
from classes import CLASS_NAMES, DISPLAY_NAMES, CLASS_COLOURS, N_CLASSES, BUILT
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("project folder:", ROOT)

In [ ]:
RUN_TRAINING = False        # True = train E1 again here (writes runs/E1_notebook, ~25 min on a GPU)
if RUN_TRAINING:
    import subprocess
    cmd = [sys.executable, "-u", str(ROOT / "train" / "train.py"), "--prep-dir", str(PREP), "--name", "E1_notebook",
           "--label", "hyb_s", "--runs-dir", str(RUNS)]
    with subprocess.Popen(cmd, cwd=ROOT / "train", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as p:
        for line in p.stdout:
            print(line, end="")

**About this cell.** Training is done with the script `train/train.py` on the server's GPU (each run 15–50 min), not
inside the notebook: a notebook would stop if the browser disconnects. The notebook reads the files the runs saved.
Setting `RUN_TRAINING = True` repeats E1 here to show that the script and the notebook give the same result.

## 2.1 The network

ResNet34 encoder (ImageNet start, fine-tuned), decoder with 5 upsampling blocks and skip connections, 1×1 convolution
to 6 class scores per pixel (`train/model.py`). Input: 3 channels = VV, VH, VH−VV in dB, normalised.

In [ ]:
import torch
from model import build_model
net = build_model("resnet34", "none").eval()
shapes = []
hooks = [m.register_forward_hook(lambda m, i, o, n=n: shapes.append((n, tuple(o.shape[1:]))))
         for n, m in [("encoder stage %d" % k, s) for k, s in enumerate([net.encoder.layer1, net.encoder.layer2,
                       net.encoder.layer3, net.encoder.layer4], 1)] + [("decoder block %d" % k, b) for k, b in enumerate(net.decoder, 1)]
                      + [("1x1 output", net.head)]]
with torch.no_grad():
    net(torch.zeros(1, 3, 256, 256))
for h in hooks: h.remove()
display(pd.DataFrame(shapes, columns=["layer", "output (channels, H, W)"]))
n_enc = sum(p.numel() for p in net.encoder.parameters()); n_all = sum(p.numel() for p in net.parameters())
print(f"parameters: {n_all / 1e6:.1f} M (encoder {n_enc / 1e6:.1f} M, decoder + head {(n_all - n_enc) / 1e6:.1f} M)")

### How and why: the U-Net

**What.** The table lists the size of the data after each block for one 256 × 256 input.

**How it works.**
* **Encoder (ResNet34)**: 4 stages of 3 × 3 convolutions; each stage halves the image size and increases the number
  of channels (feature maps): 256 → 128 → 64 → 32 → 16 → 8 pixels, 64 → 512 channels. Early layers detect edges and
  speckle texture; deep layers see a whole neighbourhood (an 8 × 8 map where each value "sees" ~250 m around it):
  "a bright, regular pattern surrounded by fields" = village.
* **Decoder**: 5 blocks, each doubles the size again (nearest-neighbour interpolation, or a learned transposed convolution in E3),
  concatenates the encoder features of the same size (**skip connections**: the fine details from early layers,
  which the deep layers have lost) and applies two 3 × 3 convolutions.
* **Head**: a 1 × 1 convolution turns the 16 decoder channels of each pixel into 6 class scores.

**Why a U-Net?** Segmentation needs both *context* (is this bright pixel a roof or a rock?) and *precision* (where
exactly is the edge of the town?). The encoder gives context, the skip connections give precision. It is the
standard architecture for satellite segmentation and works with small datasets.

**Why ResNet34 and ImageNet weights?** ResNet34 is deep enough to learn context but small enough (24 M parameters)
for 400 training places. ImageNet weights are a *transfer learning* start: the first layers already detect edges
and textures. Radar is not a photo, so the whole network is fine-tuned (all weights trained), not frozen. Our three
channels (VV, VH, VH − VV) replace the three colour channels (R, G, B), so the pretrained first layer fits.

### What goes in and what comes out

One test scene through the trained E1: the input channels, the 6 class scores (logits) per pixel, the softmax
probabilities, and the final map = the class with the highest probability.

In [ ]:
from pipeline import load_model, colourise
model, stats, cfg, device = load_model(RUNS / "E1")
idx = pd.read_csv(PREP / "index.csv"); tt = idx[(idx.status == "ok") & (idx.split == "test")]
zz = np.load(PREP / f"{tt[tt.settlement.isin(['town', 'city'])].iloc[0].tile_id if (tt.settlement.isin(['town', 'city'])).any() else tt.iloc[0].tile_id}.npz")
k = int(zz["best"][zz["best"] >= 0][0]); sc = zz["s1"][k]
mean = np.asarray(stats["mean"], np.float32)[:, None, None]; std = np.asarray(stats["std"], np.float32)[:, None, None]
x = (sc.astype(np.float32) / 100 - mean) / std; x[sc == -32768] = 0
with torch.no_grad():
    logits = model(torch.from_numpy(x)[None].to(device)).float().cpu()[0]
prob = logits.softmax(0).numpy()
print(f"input {tuple(x.shape)} -> logits {tuple(logits.shape)} -> softmax (sums to 1 per pixel) -> argmax map {prob.shape[1:]}")
c0 = prob.shape[1] // 2
print(f"one pixel ({c0}, {c0}):")
display(pd.DataFrame({"logit": [round(float(v), 2) for v in logits[:, c0, c0]],
                      "probability": [round(float(v), 3) for v in prob[:, c0, c0]]}, index=DISPLAY_NAMES).T)
print("-> predicted class:", DISPLAY_NAMES[int(prob[:, c0, c0].argmax())])
fig, ax = plt.subplots(2, 6, figsize=(19, 6.6))
for b, nme in enumerate(["VV", "VH", "VH-VV"]):
    ax[0, b].imshow(x[b], cmap="gray", vmin=-2.5, vmax=2.5); ax[0, b].set_title(f"input: {nme} (normalised)", fontsize=9)
ax[0, 3].imshow(colourise(zz["hyb_s"][int(zz["month"][k])])); ax[0, 3].set_title("label", fontsize=9)
ax[0, 4].imshow(colourise(prob.argmax(0))); ax[0, 4].set_title("prediction (argmax)", fontsize=9)
ax[0, 5].imshow(prob.max(0), cmap="viridis", vmin=0.2, vmax=1); ax[0, 5].set_title("confidence (max probability)", fontsize=9)
for c in range(6):
    ax[1, c].imshow(prob[c], cmap="viridis", vmin=0, vmax=1); ax[1, c].set_title(f"P({DISPLAY_NAMES[c]})", fontsize=9)
for a in ax.ravel(): a.axis("off")
plt.tight_layout(); plt.show()

### How and why: from input to map

**Reading the output.** Top row: the three normalised input channels, the label, the predicted map and the
confidence (highest probability; dark = the network is unsure). Bottom row: the probability of each class per pixel.
The small table shows one pixel: 6 **logits** (raw scores, any size), turned into **probabilities** by softmax
(exp(score) / sum of exp(scores): positive, sum to 1), and the class with the highest probability is the answer.

**What to notice.** The network is confident inside large uniform areas and unsure along borders and in mixed areas
(gardens between houses, fields with scattered trees). The prototype uses these probabilities, not only the final
class: for built-up change it requires a *high* built-up probability at the end and a *low* one at the start.

## 2.2 The variants

| Run | What changes vs E1 | Question |
| --- | --- | --- |
| E1 | main model: hybrid + smoothed monthly labels, ImageNet fine-tuned, interpolation upsampling | |
| E0 | raw Dynamic World labels | do building footprints + seasonal smoothing help? |
| E1y | annual label (yearly majority) | are stable annual labels better than monthly ones? |
| E2a | ImageNet encoder frozen | transfer learning as feature extraction (lecture 08) |
| E2b | no pretraining (from scratch) | does ImageNet help radar at all? |
| E3 | transposed-convolution upsampling | learned "inverse of convolution" vs interpolation (lecture 07) |
| E5 | 5×5 Lee speckle filter on every scene | does speckle filtering help raw single scenes? |
| E1half | only half of the training places | is the dataset big enough? (learning curve) |
| E1s1, E3s1 | E1 / E3 with another random seed | how large is run-to-run noise? |
| E6 | E3 + Dropout 0.2 + random ±1.5 dB VV/VH offset and 0.5 dB noise + Lovász loss | does regularisation against moisture / overfitting help? |

Common setup: CE + Dice loss with class weights, AdamW (lr 1e-3, cosine schedule), batch 32, bf16, random 90°
rotations and flips, balanced sampler, early stopping after 10 epochs without val mIoU gain.

In [ ]:
ORDER = ["E1", "E1s1", "E0", "E1y", "E2a", "E2b", "E3", "E3s1", "E5", "E6", "E1half"]
runs = [r for r in ORDER if (RUNS / r / "test_metrics.json").exists()]
print("finished runs:", runs)

### How and why: the training setup common to all runs

| Setting | Value | Why |
| --- | --- | --- |
| Loss | weighted cross-entropy + Dice | cross-entropy trains every pixel; Dice optimises overlap per class, so small classes (built-up, water) are not drowned by large ones |
| Optimizer | AdamW, learning rate 0.001, weight decay 0.0001 | Adam adapts the step size per weight; weight decay keeps weights small (less overfitting) |
| Schedule | 1 warm-up epoch, then cosine decay to 0 | a gentle start protects the pretrained weights; small steps at the end fine-tune |
| Batch | 32 scenes | as large as GPU memory allows, smoother updates |
| Precision | bfloat16 mixed precision | about 2× faster, no accuracy loss |
| Augmentation | random 90° rotations + flips | fields and towns have no "up" |
| Epochs | up to 40, early stopping after 10 without improvement | stops before overfitting; the best epoch's weights are kept |
| Sampler | balanced (notebook 01, 1.4) | every region and settlement type equally |
| Labels ignored | pixels without label or radar data | the loss is computed only where the truth is known |

**How the variants were chosen.** Each variant changes **one** thing compared to E1, so a difference can be
attributed to that change (a controlled experiment). Second runs with another random seed (E1s1, E3s1) measure
how much results differ by chance alone; only differences larger than that count.

## 2.3 Training logs (as printed during training)

In [ ]:
for r in runs:
    log = ROOT / "train" / f"runs_{r}.log"
    if log.exists():
        text = log.read_text()
        text = text[text.find(f"=== {r}"):] if f"=== {r}" in text else text
        print(text.split("Saved to")[0].strip()[-6000:], "\n" + "-" * 100)

**Reading the logs.** Each run first prints its setup (scenes, sampler, model size, class weights), then one line per
epoch: `loss train/val`, `mIoU train/val`, accuracy, seconds, and `*best` when validation mIoU improved (that
checkpoint is saved), and at the end the test results. **Notebook 04 turns these logs into tables and charts.**

## 2.4 Training curves

In [ ]:
ncol = 6
nrow = -(-len(runs) // ncol)
fig, axes = plt.subplots(2 * nrow, ncol, figsize=(3.2 * ncol, 3.0 * 2 * nrow), squeeze=False)
for a in axes.ravel(): a.axis("off")
for j, r in enumerate(runs):
    h = pd.read_csv(RUNS / r / "history.csv")
    best = h.val_miou.idxmax()
    al, am = axes[2 * (j // ncol), j % ncol], axes[2 * (j // ncol) + 1, j % ncol]
    for a in (al, am): a.axis("on")
    al.plot(h.epoch, h.train_loss, label="train"); al.plot(h.epoch, h.val_loss, label="val")
    am.plot(h.epoch, h.train_miou, label="train"); am.plot(h.epoch, h.val_miou, label="val")
    am.axvline(h.epoch[best], color="grey", ls=":", lw=1)
    al.set_title(r); am.set_ylim(0.3, 0.85); am.set_xlabel("epoch")
    if j % ncol == 0:
        al.set_ylabel("loss"); am.set_ylabel("mIoU")
axes[0, 0].legend(); axes[1, 0].legend()
plt.tight_layout(); plt.show()
print("Dotted line = best validation epoch (checkpoint used for testing). Train keeps improving while val flattens:")
print("the model starts to memorise training places and label noise; early stopping keeps the best epoch.")

**How to read training curves.** Top: loss (lower = better); bottom: mIoU (higher = better); blue = training,
orange = validation.
* Both falling / rising together = the network is learning general rules.
* Training still improving while validation is flat or worse = **overfitting** (memorising training places).
* The dotted line = best validation epoch; that checkpoint is used for testing (early stopping).
* Validation loss can rise while validation mIoU still improves: the loss also punishes *confident* mistakes,
  mIoU only counts right / wrong.

**What we learned.** All runs reach their best validation mIoU after 11–24 epochs; training mIoU keeps rising
afterwards, so without early stopping the models would overfit. The frozen encoder (E2a) stops earliest and scores lowest on the test places (0.462): radar
features differ from photo features, so the encoder must adapt.

## 2.5 Test results (130 unseen places)

Each run scored against its own training label, with one scene per month (as printed at the end of training).

In [ ]:
rows = []
for r in runs:
    m = json.loads((RUNS / r / "test_metrics.json").read_text())
    o, b = m["overall"], m.get("built", {})
    rows.append({"run": r, "OA": o["OA"], "mIoU": o["mIoU"], "macro F1": o["macro_F1"],
                 "built P": b.get("precision"), "built R": b.get("recall"),
                 "village built R": m.get("village_built", {}).get("recall"),
                 "flip rate": m.get("flip_rate", {}).get("prediction"), "best epoch": m.get("best_epoch")})
test = pd.DataFrame(rows).set_index("run")
display(test.round(3))
iou = pd.DataFrame({r: json.loads((RUNS / r / "test_metrics.json").read_text())["per_class_iou"] for r in runs})
iou.index = DISPLAY_NAMES
display(iou.round(3).rename_axis("IoU per class"))

### How and why: the test results

**What.** Each run's best checkpoint was evaluated once on the 130 test places, **every scene** against its month's
label (each place weighted equally). Per-class IoU below.

**Reading the columns.** OA = share of correct pixels; mIoU = mean IoU over 6 classes (the main score); built P / R =
built-up precision and recall; village built R = how much village built-up is found; flip rate = month-to-month
change of the predicted maps (one scene per month); best epoch.

**Careful.** Here each run is scored against **its own** training label (E0 against raw Dynamic World, E1 against
smoothed hybrid, ...), so the rows are not directly comparable. Section 2.6 compares all runs on the same labels.

**What we learned.** Single-scene mIoU is around 0.48–0.50 for all good variants: the differences are within the
run-to-run noise measured with second seeds (± 0.01 overall, ± 0.05 per region). The IoU per class shows the pattern
of notebook 01: water and built-up best, grass & shrub / crops / bare lowest.

In [ ]:
MAIN = "E1"
m = json.loads((RUNS / MAIN / "test_metrics.json").read_text())
f1 = pd.Series(m["per_class_f1"]); f1.index = DISPLAY_NAMES
cm = pd.read_csv(RUNS / MAIN / "test_confusion.csv", index_col=0).to_numpy(float)
cmn = cm / cm.sum(1, keepdims=True)
fig, ax = plt.subplots(1, 3, figsize=(17, 4.6), gridspec_kw={"width_ratios": [1.2, 1, 1]})
im = ax[0].imshow(cmn, cmap="Blues", vmin=0, vmax=1)
ax[0].set_xticks(range(6), DISPLAY_NAMES, rotation=35, ha="right"); ax[0].set_yticks(range(6), DISPLAY_NAMES)
for i in range(6):
    for j in range(6):
        ax[0].text(j, i, f"{cmn[i, j]:.2f}", ha="center", va="center", fontsize=8, color="w" if cmn[i, j] > .5 else "k")
ax[0].set_xlabel("predicted"); ax[0].set_ylabel("label"); ax[0].set_title(f"{MAIN}: confusion (row-normalised)")
pd.Series({k: v["mIoU"] for k, v in m["per_region"].items()}).sort_values().plot.barh(ax=ax[1], color="#419bdf")
ax[1].set_title("mIoU per world region"); ax[1].set_xlim(0, 0.7)
pd.Series({k: v["mIoU"] for k, v in m["per_settlement"].items()}).plot.bar(ax=ax[2], rot=0, color="#e49635")
ax[2].set_title("mIoU per settlement type"); ax[2].set_ylim(0, 0.7)
plt.tight_layout(); plt.show()
tp = np.diag(cm); prec = tp / np.maximum(cm.sum(0), 1); rec = tp / np.maximum(cm.sum(1), 1)
f1c = 2 * prec * rec / np.maximum(prec + rec, 1e-9); sup = cm.sum(1)
rep_ = pd.DataFrame({"precision": prec, "recall": rec, "f1-score": f1c, "support (pixels)": sup}, index=DISPLAY_NAMES)
rep_.loc["macro avg"] = [prec.mean(), rec.mean(), f1c.mean(), sup.sum()]
rep_.loc["weighted avg"] = [np.average(prec, weights=sup), np.average(rec, weights=sup), np.average(f1c, weights=sup), sup.sum()]
display(Markdown(f"**{MAIN}: classification report** (as `sklearn.metrics.classification_report` in the labs, pixel level, "
                 f"each place weighted equally; accuracy = {tp.sum() / cm.sum():.3f})"))
display(rep_.round(3).astype({"support (pixels)": int}))

### How and why: confusion matrix, regions, settlement types

**Confusion matrix** (left): row = true class, column = predicted class, each row sums to 1. The diagonal is the
recall of each class. Off-diagonal cells show which classes are mixed up, e.g. grass & shrub ↔ crops ↔ trees: these
are radar-specific confusions (similar structure and moisture), not a bug.

**Per region and settlement type** (middle, right): this answers *"does it work on places unlike the training
data?"*. Every test place lies in a 5° block without training places, so these are all new places. Differences
between regions come from the landscapes (deserts and mountains are hard for radar) and from label quality (Dynamic
World is worse in the tropics, where clouds are frequent).

**Classification report** (table): precision, recall and F1 per class, the same format as
`sklearn.metrics.classification_report` in the lab exercises; *support* = number of labelled test pixels.

## 2.6 From single scenes to 12 monthly maps

The model reads one raw scene at a time. The prototype needs one map per month; we compared four ways of combining a
month's scene predictions (`train/eval_change.py`), all scored against the **same** labels (hyb_s) so the runs are
comparable. *Month mean* = class probabilities averaged over all scenes of the month.

In [ ]:
import subprocess
RUN_MISSING_EVAL = True     # runs eval_change.py (~5 min per run on a GPU) where change_eval.csv is missing
for r in runs:
    if RUN_MISSING_EVAL and not (RUNS / r / "change_eval.csv").exists():
        print(f"evaluating {r} ...")
        subprocess.run([sys.executable, str(ROOT / "train" / "eval_change.py"), "--run", str(RUNS / r),
                        "--prep-dir", str(PREP), "--label", "hyb_s"], cwd=ROOT / "train", check=False)
ce = {r: pd.read_csv(RUNS / r / "change_eval.csv", index_col=0) for r in runs if (RUNS / r / "change_eval.csv").exists()}
if "E1" in ce:
    display(Markdown("**E1: four ways to build the monthly maps**")); display(ce["E1"].round(3))
cols = ["OA", "mIoU", "built_P", "built_R", "flip_rate", "built_false_change", "false_new_ha_km2"]
fair = pd.DataFrame({r: d.loc[d.index.str.startswith("B")].iloc[0] for r, d in ce.items()}).T
if len(fair):
    display(Markdown("**All runs, month-mean maps, same reference labels (hyb_s): the fair comparison**"))
    display(fair[cols].round(3))

### How and why: from single scenes to 12 monthly maps

**What.** The prototype needs one map per month, but each month has 2–15 scenes. Four ways to combine them were
compared (`train/eval_change.py`):
* **A best scene**: only the scene with the best coverage nearest mid-month;
* **D best scene, one orbit**: the same, but always from the same orbit direction (radar looks from a different side
  on ascending and descending passes);
* **B month mean**: class probabilities of *all* scenes of the month averaged, then argmax;
* **C month mean, one orbit**.

**Why averaging helps.** Each scene has its own speckle, soil moisture and look direction; their errors are partly
random, so averaging cancels part of them. It is the same reason an ensemble of models helps.

**Reading the tables.** The first table: E1 with the four methods. The second: **all runs with method B against the
same label (hyb_s)**: this is the **fair comparison** of the variants. Columns: flip_rate = monthly flicker;
built_false_change = how often built-up status flips where the label is stable; false_new_ha_km2 = new built-up
reported per km² per year that is not new in the labels.

**What we learned.** Month mean (B) is clearly best: mIoU 0.487 → 0.522 and flicker 25 % → 15 % for E1. Restricting to
one orbit did not help. On the same labels, E1 / E3 / E3s1 / E5 are within ~0.01 of each other: no single training
trick is a big win; how the scenes are combined matters much more.

In [ ]:
if len(fair):
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
    fair["mIoU"].plot.bar(ax=ax[0], rot=0, color="#419bdf"); ax[0].set_ylim(0.4, 0.6); ax[0].set_title("mIoU (month mean)")
    (fair["flip_rate"] * 100).plot.bar(ax=ax[1], rot=0, color="#e49635"); ax[1].set_title("month-to-month flicker (%)")
    fair["false_new_ha_km2"].plot.bar(ax=ax[2], rot=0, color="#c4281b"); ax[2].set_title("false new built-up (ha/km² per year)")
    plt.tight_layout(); plt.show()
    print("Differences of about +-0.02 mIoU are within run-to-run noise.")

## 2.7 Is the dataset big enough? (learning curve)

In [ ]:
if "E1half" in ce and "E1" in ce:
    lc = pd.DataFrame({"E1half (50% of training places)": ce["E1half"].iloc[2], "E1 (100%)": ce["E1"].iloc[2]}).T
    display(lc[cols].round(3))
elif "E1half" in runs:
    display(test.loc[[r for r in ["E1half", "E1"] if r in runs]])
else:
    print("E1half not trained yet:  python3 train.py --prep-dir ../data/tiles_v0/prepared --name E1half --label hyb_s --train-frac 0.5")

### How and why: is the dataset big enough?

**What.** E1half is E1 trained with only half of the training places (same share in every region × settlement type;
validation and test unchanged).

**Why.** If half the data gives the same result, more data would not help and effort should go elsewhere. If half
the data is clearly worse, more places would probably help further.

**What we learned.** Halving the places cost about 2.4 mIoU points (0.522 → 0.498 month mean). So the dataset was
doubled from 300 to 600 places; more places would likely help again. A fellow student's project found the opposite
for *more years of the same places*, consistent with this: **variety of places** matters more than more images of
the same places.

## 2.8 Final setting for the prototype: ensemble and change rule

No retraining (`train/eval_ensemble.py`): the probabilities of the 4 best models are averaged (ensemble), test-time
augmentation (flipped copies) is tried, and two rules for **new built-up** are compared: *argmax* (built in most end
months, in none of the start months) and *confident* (mean built-up probability ≥ HI at the end and < LO in every
start month).

In [ ]:
f = RUNS / "ensemble_eval.csv"
if f.exists():
    ee_ = pd.read_csv(f)
    display(Markdown("**Maps**"))
    display(ee_[ee_.change_rule == "argmax"].set_index("variant")[["OA", "mIoU", "built_P", "built_R", "flip_rate", "built_false_change"]].round(3))
    display(Markdown("**New built-up over 12 months** (false = not new in the labels; the labels themselves are noisy)"))
    display(ee_.set_index(["variant", "change_rule"])[["new_ha_km2", "false_new_ha_km2", "change_P"]].round(3))
    try:
        from pipeline import PROTOTYPE
    except ImportError:   # older prototype/pipeline.py on this machine
        PROTOTYPE = {"runs": ["E3", "E1", "E5", "E2b"], "tta": False, "change_rule": "confident", "hi": 0.5, "lo": 0.2}
    display(Markdown(f"**Prototype uses:** models {' + '.join(PROTOTYPE['runs'])}, TTA {PROTOTYPE['tta']}, "
                     f"{PROTOTYPE['change_rule']} rule (HI {PROTOTYPE['hi']}, LO {PROTOTYPE['lo']}). The ensemble adds about "
                     "+2 mIoU points and less flicker; TTA adds almost nothing at 4x the time; the confident rule removes "
                     "almost all false new built-up but is strict, so the app also shows the argmax result as an upper bound."))
else:
    print("ensemble_eval.csv not found: run train/eval_ensemble.py")

### How and why: the final setting

**Ensemble.** The probabilities of 4 models (E3, E1, E5, E2b) are averaged. They were trained with different
settings, so they make partly different mistakes, which average out: +2 mIoU points, less flicker. Cost: 4× the
computing time, acceptable for a 5 km area.

**TTA (test-time augmentation).** Each scene is also predicted flipped (horizontally, vertically, both) and the
results averaged. Almost no gain at 4× the time: the month mean already averages many scenes. Not used.

**Change rules.** *argmax*: a pixel is new built-up if it is built-up in most of the last 3 months and in none of the
first 3. *confident*: the mean built-up probability at the end ≥ 0.5 **and** below 0.2 in every start month.
Patches smaller than 4 pixels (400 m²) are dropped in both. "false new" = new built-up not confirmed by the labels.

**What we learned.** The confident rule removes almost all false new built-up (≈ 0.005 ha/km² per year on test
places instead of ≈ 0.4 with argmax), but in the gallery (notebook 03) it also missed obvious growth (Lagos 0.5 ha).
A *moderate* rule (below 0.35 instead of 0.2 at the start) reported ~0 ha in the stable places and tens of hectares
where both references see growth, so it became the headline rule; the app shows confident and argmax as the lower and
upper bound.

## 2.9 Round 2: run-to-run noise, regularisation (E6), 3-month smoothing

Three follow-ups after comparing with a fellow student's approach (ideas only): repeat runs with a second seed,
a regularised model (E6), and averaging each month's probabilities with its neighbours ("smooth3").

In [ ]:
if all((RUNS / r / "change_eval.csv").exists() for r in ["E1", "E1s1", "E3", "E3s1"]):
    seeds = pd.DataFrame({r: pd.read_csv(RUNS / r / "change_eval.csv", index_col=0).iloc[2] for r in ["E1", "E1s1", "E3", "E3s1", "E6"]
                          if (RUNS / r / "change_eval.csv").exists()}).T
    display(Markdown("**Same settings, different random seed (month-mean maps):** differences = run-to-run noise"))
    display(seeds[["OA", "mIoU", "built_P", "built_R", "flip_rate", "false_new_ha_km2"]].round(3))
for fn, title in [("ensemble_eval_round2.csv", "3-month smoothing (ensemble of E3, E1, E5, E2b)"),
                  ("ensemble_eval_with_E6.csv", "Adding E6 to the ensemble")]:
    f = RUNS / fn
    if f.exists():
        d = pd.read_csv(f)
        display(Markdown(f"**{title}**"))
        display(d[d.change_rule == "argmax"].set_index("variant")[["OA", "mIoU", "built_P", "built_R", "flip_rate", "built_false_change"]].round(3))
        display(d[d.change_rule.isin(["confident 0.5/0.2", "confident 0.5/0.35"])].set_index(["variant", "change_rule"])[["new_ha_km2", "false_new_ha_km2", "change_P"]].round(3))
display(Markdown("**Reading:** a second seed moves mIoU by about 0.01 and the argmax false new built-up by up to 0.17 ha/km², "
                 "so differences of that size between variants are noise. E6 did not help (mIoU 0.501 vs 0.522-0.527, "
                 "bare & snow IoU 0.36 -> 0.26): absolute backscatter level carries class information that the brightness "
                 "shifts destroy. 3-month smoothing halves the flicker (11.6% -> 5.9%) and slightly raises mIoU, so the "
                 "prototype uses ensemble + smooth3; built-up change uses the moderate rule (0.5 / 0.35, chosen on the "
                 "gallery places, notebook 03)."))

### How and why: round 2 (run-to-run noise, E6, smoothing)

**Second seeds (E1s1, E3s1).** Training has randomness: the order of samples, augmentation, the random start of the
decoder. Training the same setting twice shows how much results vary by chance: about ± 0.01 mIoU overall and up to
± 0.05 per region. Earlier claims like "E3 has 25 % less false change" turned out to be partly this noise.

**E6 (regularisation).** E3 + Dropout 0.2 + random brightness shifts of ± 1.5 dB per scene + 0.5 dB pixel noise +
Lovász loss (a direct IoU optimiser). Idea: make the network robust to soil moisture and rain. Result: worse (mIoU
0.501 vs 0.522–0.527), especially bare & snow (IoU 0.36 → 0.26). Lesson: the absolute backscatter level carries class
information (dry sand is very dark, wet soil brighter); shifting brightness at random destroys that information.

**smooth3.** Each month's probabilities averaged with the previous and next month (not wrapped around the year).
Flicker 11.6 % → 5.9 % with slightly higher mIoU (0.544 → 0.549). Real land-use change lasts, so a 3-month window
removes mostly noise. This is the setting the prototype uses.

## 2.10 Example predictions on test places

Label, one scene per month, and the month-mean map (what the prototype shows).

In [ ]:
from pipeline import load_model, colourise
from eval_change import scene_probs, monthly_maps
model, stats, cfg, device = load_model(RUNS / "E1")
mean = np.asarray(stats["mean"], np.float32)[:, None, None]; std = np.asarray(stats["std"], np.float32)[:, None, None]
idx = pd.read_csv(PREP / "index.csv"); test_tiles = idx[(idx.status == "ok") & (idx.split == "test")]
def well_labelled(df, min_frac=0.5):
    """first place of df that has at least one month with >= min_frac labelled pixels and radar data"""
    for r in df.itertuples():
        zz_ = np.load(PREP / f"{r.tile_id}.npz")
        if any(zz_["n"][m] > 0 and (zz_["hyb_s"][m] >= 0).mean() >= min_frac for m in range(12)):
            return df[df.tile_id == r.tile_id]
    return df.head(1)
picks = pd.concat([well_labelled(test_tiles[test_tiles.settlement == s]) for s in ["city", "town", "village", "none"]])
fig, axes = plt.subplots(len(picks), 4, figsize=(13, 3.3 * len(picks)), squeeze=False)
for i, r in enumerate(picks.itertuples()):
    z = np.load(PREP / f"{r.tile_id}.npz")
    probs = scene_probs(model, z["s1"], mean, std, torch.device(device), bool(cfg.get("lee", False)))
    nod = (z["s1"] == -32768).any(1)
    a = monthly_maps(probs, nod, z["month"].astype(int), z["day"].astype(int), z["valid"], z["orbit"].astype(str), "best", False)
    b = monthly_maps(probs, nod, z["month"].astype(int), z["day"].astype(int), z["valid"], z["orbit"].astype(str), "mean", False)
    labelled = np.array([(z["hyb_s"][mm] >= 0).mean() if z["n"][mm] > 0 else -1 for mm in range(12)])
    m = int(np.argmax(labelled))          # the month with the most labelled pixels (and radar data)
    vv = z["s1"][z["best"][m], 0].astype(float) / 100
    axes[i, 0].imshow(vv, cmap="gray", vmin=-25, vmax=0); axes[i, 1].imshow(colourise(z["hyb_s"][m]))
    axes[i, 2].imshow(colourise(a[m])); axes[i, 3].imshow(colourise(b[m]))
    axes[i, 0].set_ylabel(f"{r.settlement}, {r.region}", fontsize=9)
    for j, t in enumerate(["S1 VV", "label", "one scene", f"mean of {int(z['n'][m])} scenes"]):
        axes[i, j].set_title(f"{t} (month {m + 1})", fontsize=9); axes[i, j].set_xticks([]); axes[i, j].set_yticks([])
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in CLASS_COLOURS]
fig.legend(handles, DISPLAY_NAMES, loc="lower center", ncol=6, frameon=False)
plt.tight_layout(rect=(0, 0.03, 1, 1)); plt.show()

### How to read the example predictions

One test place per settlement type (city, town, village, none). Columns: radar VV, label, prediction from **one**
scene, prediction from the **mean of all scenes** of that month. The month-mean map is smoother and closer to the
label, especially at class borders; the single-scene map has scattered wrong pixels (speckle). Remaining errors are
mostly between grass & shrub, crops and bare, and small houses in villages that are missed (a single-storey house
of 8 × 8 m covers less than one 10 m pixel, and Sentinel-1's real resolution is about 20 m).